# Larger LLMs for schema discovery, smaller LLMs for application

We build one **Elastic Workflow** that asks: *what does this report reveal about the pilot who wrote it?*
A large reasoning model reads a stratified sample of [NASA ASRS](https://asrs.arc.nasa.gov/) reports and proposes a categorical schema. A human approves it from the Kibana UI. A small [Mistral](https://mistral.ai/) model applies the schema across the full corpus.

To be clear about the question: ASRS reports are anonymous, so "about the pilot" does not mean identity or credentials. It refers to behavioral traits that show up in how the incident is narrated, such as decision-making style, situational awareness, and accountability. Which traits exactly? That is precisely what the discovery step decides by reading the sample.

Why split the work this way? Running a large reasoning model over every document is slow and expensive. Instead, we pay for deep reasoning once to design the schema from a small sample, and delegate the thousands of repetitive per-document classifications to a small model. The result is close to large-model quality at small-model cost.

## Prerequisites

- Elastic Stack 9.4+ or Elastic Cloud Serverless. Workflows has been GA since 9.4.
- [Agent Builder](https://www.elastic.co/docs/solutions/search/elastic-agent-builder) enabled in your deployment.
- A [Kibana GenAI connector](https://www.elastic.co/docs/reference/kibana/connectors-kibana/ai-connector) pointing at Claude Sonnet (or an equivalent reasoning model). This is the planner.
- A [Mistral API key](https://console.mistral.ai/api-keys). We use it to register an Elasticsearch inference endpoint.
- Python 3.10+. Dependencies are pinned in [`requirements.txt`](requirements.txt) and installed by the `%pip install` cell below.

## Setup

In [ ]:
%pip install -qr requirements.txt

In [15]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
KIBANA_URL = os.getenv("KIBANA_URL")
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")

## Create the corpus index

The index mappings match the ASRS columns we will use: `flight_phase` and `anomaly` for stratification, `narrative` and `synopsis` as document text.

In [16]:
from elasticsearch import Elasticsearch, helpers

es = Elasticsearch(
    ELASTICSEARCH_URL,
    api_key=ELASTICSEARCH_API_KEY,
)

print(f"Connected to Elasticsearch: {es.info()['version']['number']}")

Connected to Elasticsearch: 9.4.2


In [ ]:
INDEX = "incident_reports"

mappings = {
    "properties": {
        "acn": {"type": "keyword"},
        "flight_phase": {"type": "keyword"},
        "anomaly": {"type": "keyword"},
        "synopsis": {"type": "text"},
        "narrative": {"type": "text"},
    }
}

if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)

es.indices.create(index=INDEX, mappings=mappings)
print(f"Created index: {INDEX}")

## Load ASRS reports

The dataset (`asrs.csv`) was extracted from the [NASA ASRS Database Online](https://asrs.arc.nasa.gov/search/database.html) using the public search interface and exported as CSV. It is bundled next to this notebook in the repository.

In [ ]:
import pandas as pd

# ASRS CSV exports have a category row above the actual column names.
# Skip the category row; pandas drops the blank line that follows automatically.
df = pd.read_csv("asrs.csv", skiprows=[0], skip_blank_lines=True, low_memory=False)

# Some column names repeat (e.g. Aircraft 1/Aircraft 2 both have "Flight Phase");
# pandas renames duplicates with a ".1" suffix. We just use the first occurrence.
df = df.dropna(subset=["ACN", "Narrative"])

# Optional columns (Flight Phase, Anomaly, Synopsis) can be empty; pandas reads
# those as NaN, which is not valid JSON, so replace them with None.
df = df.where(pd.notna(df), None)


def to_doc(row):
    return {
        "_index": INDEX,
        "_id": str(row["ACN"]),
        "_source": {
            "acn": str(row["ACN"]),
            "flight_phase": row.get("Flight Phase"),
            "anomaly": row.get("Anomaly"),
            "synopsis": row.get("Synopsis"),
            "narrative": row.get("Narrative"),
        },
    }


# raise_on_error=False: a handful of rows in the public ASRS export are
# malformed. For this demo we skip those documents and report the count below
# instead of aborting the whole ingest; drop the flag if you prefer to fail
# fast and inspect the offending rows.
ok, errors = helpers.bulk(
    es,
    (to_doc(r) for _, r in df.iterrows()),
    raise_on_error=False,
)
print(f"indexed {ok}, errors {len(errors)}")

## Register the Mistral inference endpoint

The classification step calls this endpoint once per document via `ai.agent`.

In [19]:
INFERENCE_ID = "mistral-small-extractor"

# Delete and re-create so the rate_limit setting picks up changes between runs.
# 6 RPM = 1 every 10s; conservative for Mistral free tier to avoid 429s.
es.options(ignore_status=[404]).inference.delete(inference_id=INFERENCE_ID)

es.inference.put(
    task_type="chat_completion",
    inference_id=INFERENCE_ID,
    inference_config={
        "service": "mistral",
        "service_settings": {
            "api_key": MISTRAL_API_KEY,
            "model": "mistral-small-latest",
            "rate_limit": {"requests_per_minute": 6},
        },
    },
)
print(f"Inference endpoint ready: {INFERENCE_ID}")

Inference endpoint ready: mistral-small-extractor


## Auxiliary indices

`schemas` stores one document per discovery run; `extractions` stores one document per source report per schema version.

We keep both separate from the corpus index instead of writing the labels back onto the report documents. `schemas` acts as an audit trail: every classification can be traced back to the exact schema version (and reviewer notes) that produced it. Keeping `extractions` separate also means we can re-run classification with a revised schema without mutating or re-indexing the source reports, since each run just adds a new set of extraction documents.

In [20]:
for aux in ["schemas", "extractions"]:
    if not es.indices.exists(index=aux):
        es.indices.create(index=aux)
        print(f"Created index: {aux}")

Created index: extractions


## Create the workflow via the Workflows API

The workflow is defined in [`workflow.yaml`](workflow.yaml) next to this notebook; we load it, substitute the environment-specific values, and register it via `POST /api/workflows`. Workflows is a Kibana feature, so its API is served by Kibana rather than Elasticsearch. The Python Elasticsearch client only talks to Elasticsearch endpoints, which is why this section uses plain `requests` calls against `KIBANA_URL` instead.

The workflow runs these steps in order:

1. **Stratified sample**: two aggregation queries pull a small sample of reports, one stratified by flight phase and one by anomaly type.
2. **Discover**: the large reasoning model proposes a categorical schema from the sample.
3. **Human gate**: execution pauses until a person approves (or edits) the schema from the Kibana UI.
4. **Store schema**: the approved schema is saved to the `schemas` index.
5. **Fetch corpus**: the reports to classify are retrieved from the corpus index.
6. **Classify (foreach)**: the small Mistral model applies the schema to each report and writes the result to the `extractions` index.

Neither model runs inside your cluster; both are hosted by an inference provider. The planner is whatever LLM your Kibana GenAI connector points at, for example the [Elastic Managed LLM](https://www.elastic.co/docs/explore-analyze/elastic-inference/eis) served through the Elastic Inference Service (EIS), or a provider connector for Amazon Bedrock, OpenAI, or Gemini. The executor (`mistral-small-latest`) is reached through the Elasticsearch inference endpoint we registered earlier, which calls Mistral's hosted API.

In [21]:
import requests

headers = {
    "Authorization": f"ApiKey {ELASTICSEARCH_API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}

# Discover the GenAI connector ID (large reasoning model) from Kibana.
# Looks for any Bedrock / OpenAI / Gemini / generic GenAI connector.
GENAI_TYPES = {".bedrock", ".gen-ai", ".gemini", ".inference"}

response = requests.get(f"{KIBANA_URL}/api/actions/connectors", headers=headers)
response.raise_for_status()

genai_connectors = [c for c in response.json() if c["connector_type_id"] in GENAI_TYPES]
if not genai_connectors:
    raise RuntimeError(
        "No GenAI connectors found. Create one in Kibana > Stack Management > Connectors."
    )

KIBANA_CONNECTOR_ID = genai_connectors[0]["id"]
print(f"Using connector: {genai_connectors[0]['name']} ({KIBANA_CONNECTOR_ID})")

Using connector: Anthropic Claude Haiku 4.5 (Anthropic-Claude-Haiku-4-5)


In [ ]:
from pathlib import Path

WORKFLOW_NAME = "Pilot Schema Discovery and Application"

# The full workflow definition lives in workflow.yaml next to this notebook.
# Only the environment-specific values are substituted here.
workflow_yaml = (
    Path("workflow.yaml")
    .read_text()
    .replace("__CORPUS_INDEX__", INDEX)
    .replace("__CONNECTOR_ID__", KIBANA_CONNECTOR_ID)
    .replace("__INFERENCE_ID__", INFERENCE_ID)
)

In [23]:
WORKFLOW_ID = "pilot-schema-discovery"
WORKFLOW_NAME = "Pilot Schema Discovery and Application"

# Delete any existing workflow with the same id (allows re-runs)
requests.delete(f"{KIBANA_URL}/api/workflows/workflow/{WORKFLOW_ID}", headers=headers)

# Bulk create. If the workflow already exists, the PUT below will update it.
response = requests.post(
    f"{KIBANA_URL}/api/workflows",
    headers=headers,
    json={"workflows": [{"id": WORKFLOW_ID, "yaml": workflow_yaml}]},
)
response.raise_for_status()

# PUT sets the display name, description, and enables the workflow.
put_response = requests.put(
    f"{KIBANA_URL}/api/workflows/workflow/{WORKFLOW_ID}",
    headers=headers,
    json={
        "name": WORKFLOW_NAME,
        "description": "Discovers a categorical schema from a stratified sample of incident reports, then applies it across the corpus with a small Mistral model.",
        "yaml": workflow_yaml,
        "enabled": True,
    },
)
put_response.raise_for_status()

print(f"Workflow ready: {WORKFLOW_ID} ({WORKFLOW_NAME})")
print(f"Open in Kibana: {KIBANA_URL}/app/workflows/{WORKFLOW_ID}")

Workflow ready: pilot-schema-discovery (Pilot Schema Discovery and Application)
Open in Kibana: https://4622216ea8cd443ead5bef0a3de05135.us-central1.gcp.cloud.es.io/app/workflows/pilot-schema-discovery


## Trigger the workflow

`POST /api/workflows/workflow/{id}/run` starts an execution and returns a `workflowExecutionId`. The run will pause at the `human_gate` step until the schema is approved from the Kibana UI (Workflows > Executions > the running one). After approval, the classification step fans out and writes to the `extractions` index.

In [24]:
response = requests.post(
    f"{KIBANA_URL}/api/workflows/workflow/{WORKFLOW_ID}/run",
    headers=headers,
    json={
        "inputs": {"goal": "What does this report reveal about the pilot who wrote it?"}
    },
)
response.raise_for_status()

execution_id = response.json()["workflowExecutionId"]
print(f"Execution started: {execution_id}")
print(
    f"Approve the schema at: {KIBANA_URL}/app/workflows/{WORKFLOW_ID}/executions/{execution_id}"
)

Execution started: 43b10959-03b7-4ff9-9930-cefa2543c333
Approve the schema at: https://4622216ea8cd443ead5bef0a3de05135.us-central1.gcp.cloud.es.io/app/workflows/pilot-schema-discovery/executions/43b10959-03b7-4ff9-9930-cefa2543c333


## Print the discover step output

`waitForInput` cannot be pre-populated from a previous step today (the API exposes only `message` and `schema`). To make the human gate practical, we poll the execution until the `discover` step completes and pretty-print its output. Copy that JSON, edit it if needed, and paste it into the `approved_fields` field of the form.

In [25]:
import time


def step_output(execution_id, step_id, timeout=120, interval=3):
    """Poll an execution until the given step completes, then return its output."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        exec_response = requests.get(
            f"{KIBANA_URL}/api/workflows/executions/{execution_id}", headers=headers
        )
        exec_response.raise_for_status()
        step = next(
            (
                s
                for s in exec_response.json().get("stepExecutions", [])
                if s.get("stepId") == step_id
                and s.get("status") in ("completed", "failed")
            ),
            None,
        )
        if step:
            step_response = requests.get(
                f"{KIBANA_URL}/api/workflows/executions/{execution_id}/step/{step['id']}",
                headers=headers,
            )
            step_response.raise_for_status()
            return step_response.json().get("output")
        time.sleep(interval)
    raise TimeoutError(f"step '{step_id}' did not complete in time")


discover = step_output(execution_id, "discover")

# Strip `why_useful` (not part of the human_gate form) and wrap in the shape
# expected by the waitForInput form so this is paste-ready.
approved_fields = [
    {
        "name": field["name"],
        "definition": field["definition"],
        "values": [
            {
                "value": v["value"],
                "definition": v["definition"],
            }
            for v in field["values"]
        ],
    }
    for field in discover["content"]["fields"]
]

print(json.dumps({"approved_fields": approved_fields, "notes": ""}, indent=2))

{
  "approved_fields": [
    {
      "name": "decision_making_pattern",
      "definition": "The characteristic way the pilot approaches problem-solving and decision-making when faced with an anomaly or unexpected situation",
      "values": [
        {
          "value": "decisive_independent",
          "definition": "Pilot makes clear decisions quickly, acts on their own authority, and commits to a course of action (e.g., 2237734: Captain decides to forgo diversion based on assessment; 2238341: Captain and FO coordinate but make independent decision to continue to ZZZ1)"
        },
        {
          "value": "collaborative_consensus_seeking",
          "definition": "Pilot actively seeks input from crew, dispatch, or other authorities before deciding; values multiple perspectives and shared responsibility (e.g., 2239459: Captain and FO discuss, then call dispatch and maintenance control for a huddle; 2238435: Captain requests FO to query ATC, then Captain steps in to demand vector

## Cleanup

In [ ]:
for idx in [INDEX, "schemas", "extractions"]:
    if es.indices.exists(index=idx):
        es.indices.delete(index=idx)

es.options(ignore_status=[404]).inference.delete(inference_id=INFERENCE_ID)

In [ ]:
requests.delete(f"{KIBANA_URL}/api/workflows/workflow/{WORKFLOW_ID}", headers=headers)
print(f"Deleted workflow: {WORKFLOW_ID}")